# Which models classify the most transactions and activities correctly?
In Colab, select **Runtime → Run all** on a CPU runtime. Upload the supplied **cct_train.zip** and **har_train.csv** when prompted. Keep the course data and trained artifacts private. Locally, run from the repository or its Code Prototypes directory.

Each task ranks candidates by the exact number of correct held-out predictions. F1 breaks exact accuracy ties. The fraud task reserves a later period for a final check. The activity leaderboard is a cross-validation development estimate. The last cell refits both winners and offers a private model download.

Evaluation reports accuracy alongside fraud-class F1 and activity macro-F1. The new experiments compare accuracy-oriented settings with settings chosen for the respective F1 measure on earlier or inner training folds. Highest development accuracy still determines export; F1 breaks ties.

In [1]:
from pathlib import Path
from collections import deque
import zipfile
import shutil
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 610
WORKERS = 2
print("scikit-learn", sklearn.__version__)
def locate(name):
    for root in (Path.cwd(), Path.cwd().parent):
        direct = root / name
        if direct.is_file():
            return direct
        data = root / "Docs and Data"
        if data.is_dir():
            match = next(data.rglob(name), None)
            if match:
                return match

if not all(locate(n) for n in ("cct_train.zip", "har_train.csv")):
    from google.colab import files
    files.upload()
fraud_path, har_path = map(locate, ("cct_train.zip", "har_train.csv"))
if not fraud_path or not har_path:
    raise FileNotFoundError("Upload both cct_train.zip and har_train.csv using their original names.")
entity_fields = ["cc_num", "acct_num", "first", "last", "dob", "merchant", "category"]
event_fields = entity_fields + ["unix_time", "trans_time", "trans_date", "amt", "lat", "long", "merch_lat", "merch_long"]
with zipfile.ZipFile(fraud_path) as archive:
    member, = [n for n in archive.namelist() if Path(n).name == "cct_train.csv"]
    with archive.open(member) as stream:
        fraud = pd.read_csv(stream, usecols=event_fields + ["is_fraud"], dtype={k:"string" for k in entity_fields})
fraud = fraud.sort_values("unix_time", kind="stable").reset_index(drop=True)
har = pd.read_csv(har_path, header=None)
print("Fraud rows:", len(fraud), "fraud rate:", fraud.is_fraud.mean())
print("Activity rows:", len(har), "predictors:", har.shape[1]-1)

scikit-learn 1.9.1
Fraud rows: 700807 fraud rate: 0.005106969536548579
Activity rows: 2442 predictors: 328


## Fraud · compare a purchase with earlier purchases
Compare the same tree on three increasingly rich feature sets. Card features describe spending and activity in the previous hour and day. The widest set adds account and purchaser history, familiarity with this merchant, category peers and address-to-merchant distance. First name, last name and birth date form a purchaser proxy. The identity audit below shows whether that proxy actually links multiple cards. Shared names alone are ambiguous.

Counts and averages exclude the current transaction and all transactions with the same timestamp. Category peers include earlier purchases across cards. A new entity has zero prior transactions and an unknown average. Historical fraud labels never enter the feature builder. For the research basis see [Correa Bahnsen et al.](https://doi.org/10.1016/j.eswa.2015.12.030).

In [4]:
def fraud_features(events):
    # Return features in input order, including when called on a shuffled batch.
    d = events.reset_index(drop=True).sort_values("unix_time", kind="stable")
    x = pd.DataFrame(index=d.index)
    x["amount"] = d.amt
    x["hour"] = d.trans_time.str.slice(0, 2).astype(int)
    lat1, lat2 = np.radians(d.lat), np.radians(d.merch_lat)
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(np.radians(d.merch_long-d.long)/2)**2
    x["home_merchant_km"] = 12742 * np.arcsin(np.sqrt(a.clip(0, 1)))

    entities = {"card":["cc_num"], "card_category":["cc_num","category"],
                "account":["acct_num"], "person":["first","last","dob"],
                "card_merchant":["cc_num","merchant"], "category":["category"], "merchant":["merchant"]}
    for prefix, keys in entities.items():
        # Collapse simultaneous events so their within-second ordering is immaterial.
        group_keys = keys + ["unix_time"]
        present = d[keys].notna().all(axis=1)
        batch = (d.loc[present].groupby(group_keys, sort=False, observed=True)
                 .amt.agg(total="sum", n="size").reset_index())
        groups = batch.groupby(keys, sort=False, observed=True)
        count = groups.n.cumsum() - batch.n
        total = groups.total.cumsum() - batch.total
        batch["count"] = count
        batch["mean"] = total / count.replace(0, np.nan)
        batch["elapsed"] = groups.unix_time.diff()
        prior = d[group_keys].merge(batch[group_keys + ["count","mean","elapsed"]],
                                    on=group_keys, how="left", sort=False, validate="many_to_one")
        prior.index = d.index
        x[prefix+"_count"] = prior["count"].fillna(0)
        x[prefix+"_amount_ratio"] = d.amt / prior["mean"].replace(0, np.nan)
        if prefix == "card":
            x["card_elapsed"] = prior["elapsed"]
    # Prefix sums and binary search make strict trailing windows inexpensive per card.
    for name in ("hour_count","day_count","day_spend"):
        x[name] = 0.
    x["reference_days_observed"] = 0.
    x["day_vs_prior28_ratio"] = np.nan
    for _, block in d.groupby("cc_num", sort=False):
        times = block.unix_time.to_numpy()
        end = times.searchsorted(times, side="left")
        hour = times.searchsorted(times-3600)
        day = times.searchsorted(times-86400)
        sums = np.r_[0., block.amt.to_numpy().cumsum()]
        x.loc[block.index, ["hour_count","day_count","day_spend"]] = np.column_stack(
            [end-hour, end-day, sums[end]-sums[day]])
        ref_start = times.searchsorted(times-29*86400, side="left")
        ref_total = sums[day]-sums[ref_start]
        coverage = np.clip((times-times[0])/86400-1, 0, 28)
        ratio = np.full(len(block), np.nan)
        np.divide(sums[end]-sums[day], ref_total/28, out=ratio,
                  where=(coverage == 28) & (ref_total > 0))
        x.loc[block.index, ["reference_days_observed","day_vs_prior28_ratio"]] = np.column_stack([coverage,ratio])
    return x.sort_index().astype("float32")

for keys in [["first","last"], ["first","last","dob"], ["acct_num"]]:
    card_counts = fraud.groupby(keys).cc_num.nunique()
    print(keys, "groups:", len(card_counts), "groups with multiple cards:", (card_counts > 1).sum())
Xf = fraud_features(fraud[event_fields])
yf = fraud.is_fraud.to_numpy(dtype=int)
basic = ["amount","hour"]
card = basic + ["card_amount_ratio","card_category_amount_ratio","hour_count","day_count","card_elapsed"]
extended = card + ["day_spend","card_count","card_merchant_count","home_merchant_km",
                   "account_count","account_amount_ratio","person_count","person_amount_ratio",
                   "category_amount_ratio"]
print("Feature sets:", len(basic), len(card), len(extended))

['first', 'last'] groups: 496 groups with multiple cards: 3
['first', 'last', 'dob'] groups: 500 groups with multiple cards: 0
['acct_num'] groups: 500 groups with multiple cards: 0
Feature sets: 2 7 16


### Select the fraud model and decision threshold
Train on all rows before each two-month development window. Find the accuracy-maximizing threshold on pooled out-of-window scores for each candidate. The September-onward holdout stays out of both threshold and model selection. Accuracy can exceed 99% even when fraud recall is zero, so inspect the false-positive and missed-fraud counts alongside it.

In [5]:
def best_threshold(y, score, objective="accuracy"):
    order = np.argsort(-score, kind="stable")
    ranked, truth = score[order], y[order]
    ends = np.flatnonzero(np.r_[ranked[:-1] != ranked[1:], True])
    tp = np.r_[0, np.cumsum(truth)[ends]]
    positives = np.r_[0, ends+1]
    correct = (len(y)-y.sum()) + 2*tp - positives
    tied = np.flatnonzero(correct == correct.max())
    f1 = 2*tp / np.maximum(1, positives+y.sum())
    if objective == "fraud_f1":
        tied = np.flatnonzero(f1 == f1.max())
        winner = tied[np.argmax(correct[tied])]
    else:
        winner = tied[np.argmax(f1[tied])]
    return np.r_[np.inf, ranked[ends]][winner]

def fraud_metrics(y, score, threshold):
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return dict(correct=int(tn+tp), accuracy=accuracy_score(y,pred), fraud_f1=f1_score(y,pred,zero_division=0),
                macro_f1=f1_score(y,pred,average="macro"), auc=roc_auc_score(y,score),
                false_alerts=fp, missed_frauds=fn, caught=tp)

def boost(leaves):
    return HistGradientBoostingClassifier(max_leaf_nodes=leaves, max_iter=140, learning_rate=.08,
                                          early_stopping=False, random_state=SEED)
fraud_models = {"Majority":(DummyClassifier(strategy="prior"),basic),
                "Amount cutoff":(None, ["amount"]),
                "Boost · purchase":(boost(15),basic), "Boost · card history":(boost(15),card),
                "Boost · extended":(boost(15),extended), "Boost · extended 31 leaves":(boost(31),extended)}
dates = pd.to_datetime(fraud.trans_date)
edges = pd.to_datetime(["2022-03-01","2022-05-01","2022-07-01","2022-09-01"])
folds = [(np.flatnonzero(dates < a), np.flatnonzero((dates >= a) & (dates < b)))
         for a,b in zip(edges[:-1], edges[1:])]
def fit_fraud(name, rows):
    model, cols = fraud_models[name]
    return clone(model).fit(Xf.iloc[rows][cols], yf[rows]) if model is not None else None

def fraud_scores(model, frame, cols):
    if model is None:
        return frame["amount"].to_numpy()
    return model.predict_proba(frame[cols])[:,1]

# A bagged forest tests a different algorithm on the same extended history.
from sklearn.impute import SimpleImputer
fraud_models["Random forest · extended"] = (
    make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                  RandomForestClassifier(n_estimators=100, max_samples=.5, min_samples_leaf=4,
                                         n_jobs=WORKERS, random_state=SEED)), extended)
base_forest = "Random forest · extended"
merchant_forest = "Random forest · extended + merchant history"
fraud_models[merchant_forest] = (fraud_models[base_forest][0], extended + ["merchant_count","merchant_amount_ratio"])
trend_forest = "Random forest · extended + spending trend"
fraud_models[trend_forest] = (fraud_models[base_forest][0], extended + ["reference_days_observed","day_vs_prior28_ratio"])
fraud_results = []
thresholds, f1_thresholds = {}, {}
for name, (_, cols) in fraud_models.items():
    scores = np.concatenate([fraud_scores(fit_fraud(name,tr), Xf.iloc[va],cols) for tr,va in folds])
    truth = yf[np.concatenate([va for _,va in folds])]
    thresholds[name] = best_threshold(truth,scores)
    f1_thresholds[name] = best_threshold(truth,scores,"fraud_f1")
    fraud_results.append(dict(model=name, **fraud_metrics(truth,scores,thresholds[name])))
fraud_board = pd.DataFrame(fraud_results).sort_values(["correct","fraud_f1"],ascending=False,kind="stable")
display(fraud_board)
fraud_winner = fraud_board.iloc[0].model


                                         model  correct  accuracy  fraud_f1  macro_f1       auc  false_alerts  missed_frauds  caught
8    Random forest · extended + spending trend   181872  0.998090  0.784920  0.891980  0.990157            51            297     635
6                     Random forest · extended   181864  0.998046  0.780247  0.889633  0.991809            56            300     632
7  Random forest · extended + merchant history   181833  0.997876  0.788871  0.893902  0.990286           178            209     723
3                         Boost · card history   181590  0.996543  0.618182  0.808223  0.976011           208            422     510
4                             Boost · extended   181455  0.995802  0.612658  0.805274  0.941205           438            327     605
5                   Boost · extended 31 leaves   181404  0.995522  0.564103  0.780926  0.902720           412            404     528
2                             Boost · purchase   181350  0.995226  0.

### Paired merchant-history experiment
The challenger adds two features across all cards at the merchant: earlier transaction count and purchase amount divided by earlier average amount. Each forest gets its own development-only threshold. The monthly table compares identical holdout rows. This period was inspected in earlier work, so these are development checks.

The separate spending-trend challenger adds observed reference duration and the last 24 hours’ spend divided by average daily spend over the preceding 28 days. The windows are [t−1 day, t) and [t−29 days, t−1 day). Inactive days count toward the average. A short history or zero reference spend yields a missing ratio.

In [6]:
# Compare the forests on the same later transactions.
# Both thresholds and the export winner were chosen before this comparison.
hold = np.flatnonzero(dates >= edges[-1])
before_hold = np.flatnonzero(dates < edges[-1])
hold_predictions, metrics, f1_metrics = {}, [], []
for name in dict.fromkeys([base_forest, merchant_forest, trend_forest, fraud_winner]):
    candidate_cols = fraud_models[name][1]
    scores = fraud_scores(fit_fraud(name, before_hold), Xf.iloc[hold], candidate_cols)
    threshold = thresholds[name]
    hold_predictions[name] = scores >= threshold
    f1_cut = f1_thresholds[name]
    f1_metrics.append(dict(model=name, threshold=f1_cut, **fraud_metrics(yf[hold],scores,f1_cut)))
    metrics.append(dict(model=name, threshold=threshold, **fraud_metrics(yf[hold], scores, threshold)))
print("Accuracy-selected development thresholds")
display(pd.DataFrame(metrics).set_index("model"))
print("Fraud-F1-selected development thresholds")
display(pd.DataFrame(f1_metrics).set_index("model"))
def paired_counts(truth, incumbent, challenger):
    was_right, now_right = incumbent == truth, challenger == truth
    fixed = int((~was_right & now_right).sum())
    lost = int((was_right & ~now_right).sum())
    return dict(rows=len(truth), fixed=fixed, lost=lost, net=fixed-lost,
                delta_pp=100*(fixed-lost)/len(truth))
def paired_by(truth, incumbent, challenger, slices):
    slices = np.asarray(slices)
    results = {"All": paired_counts(truth, incumbent, challenger)}
    for key in pd.unique(slices):
        mask = slices == key
        results[key] = paired_counts(truth[mask], incumbent[mask], challenger[mask])
    return pd.DataFrame.from_dict(results, orient="index")
display(paired_by(yf[hold], hold_predictions[base_forest], hold_predictions[merchant_forest],
                  dates.iloc[hold].dt.strftime("%Y-%m")))
print("Spending trend versus original forest")
display(paired_by(yf[hold], hold_predictions[base_forest], hold_predictions[trend_forest],
                  dates.iloc[hold].dt.strftime("%Y-%m")))
cols = fraud_models[fraud_winner][1]
print("Development-selected fraud model:", fraud_winner)

Accuracy-selected development thresholds
                                             threshold  correct  accuracy  fraud_f1  macro_f1       auc  false_alerts  missed_frauds  caught
model                                                                                                                                       
Random forest · extended                      0.297399   140272  0.998818  0.819172  0.909290  0.989406            35            131     376
Random forest · extended + merchant history   0.341794   140304  0.999046  0.856838  0.928179  0.987966            28            106     401
Random forest · extended + spending trend     0.349480   140270  0.998804  0.816594  0.907997  0.982823            35            133     374
Fraud-F1-selected development thresholds
                                             threshold  correct  accuracy  fraud_f1  macro_f1       auc  false_alerts  missed_frauds  caught
model                                                                   

## Activity recognition · compare linear, kernel, neighbor and tree models
Group identical feature vectors before stratification so a repeated sensor reading cannot appear in both training and validation. Fit scaling and constant-column removal inside each fold. The two logistic models measure the effect of class weighting on accuracy. The neighbor model tests whether proximity to similar sensor windows helps. See [Jordão et al. on activity-recognition evaluation](https://arxiv.org/abs/1806.05226).

In [2]:
Xh, yh = har.iloc[:,:-1], har.iloc[:,-1].to_numpy()
groups = pd.util.hash_pandas_object(Xh, index=False).to_numpy()
labels = np.unique(yh)
# Accept a split only if all activities occur on both sides of every fold.
for split_seed in range(SEED, SEED+200):
    splitter = StratifiedGroupKFold(4, shuffle=True, random_state=split_seed)
    har_folds = list(splitter.split(Xh, yh, groups))
    if all(len(np.unique(yh[tr])) == len(labels) == len(np.unique(yh[va])) for tr,va in har_folds):
        break
else:
    raise ValueError("No four-fold split covers all activities.")
print("Fold seed:", split_seed)
def scaled(model):
    return make_pipeline(VarianceThreshold(), StandardScaler(), model)
har_models = {
    "Majority": DummyClassifier(strategy="most_frequent"),
    "Logistic": scaled(LogisticRegression(max_iter=2500)),
    "Balanced logistic": scaled(LogisticRegression(max_iter=2500,class_weight="balanced")),
    "RBF SVM C=1": scaled(SVC(C=1)),
    "RBF SVM C=10": scaled(SVC(C=10)),
    "Distance-weighted neighbors": scaled(KNeighborsClassifier(n_neighbors=5,weights="distance")),
    "Random forest": RandomForestClassifier(n_estimators=240,n_jobs=WORKERS,random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(n_estimators=240,n_jobs=WORKERS,random_state=SEED)}
har_results, predictions = [], {}
for name, candidate in har_models.items():
    pred = np.empty_like(yh)
    for tr,va in har_folds:
        assert not np.intersect1d(groups[tr],groups[va]).size
        pred[va] = clone(candidate).fit(Xh.iloc[tr],yh[tr]).predict(Xh.iloc[va])
    predictions[name] = pred
    har_results.append(dict(model=name, correct=(pred == yh).sum(), accuracy=accuracy_score(yh,pred),
                            macro_f1=f1_score(yh,pred,average="macro")))
har_board = pd.DataFrame(har_results).sort_values(["correct","macro_f1"],ascending=False,kind="stable")
display(har_board)


Fold seed: 610
                         model  correct  accuracy  macro_f1
6                Random forest     1843  0.754709  0.478000
7                  Extra Trees     1806  0.739558  0.461880
4                 RBF SVM C=10     1805  0.739148  0.527703
3                  RBF SVM C=1     1738  0.711712  0.431320
1                     Logistic     1731  0.708845  0.569396
2            Balanced logistic     1694  0.693694  0.558408
5  Distance-weighted neighbors     1484  0.607699  0.422367
0                     Majority      696  0.285012  0.040327


### Which forest errors does the SVM recover?
The diagnostic uses known validation labels to identify complementary errors. Test whether a fixed two-to-one forest/SVM probability blend realizes a gain without those labels. The weights and SVM settings are fixed before scoring. Only the outer training rows enter probability calibration.

For the tuned challenger, pick one of five forest/SVM ratios by pooled inner-fold accuracy. Duplicate groups stay together in both weight selection and probability calibration. The two blends share identical fitted estimators in each outer fold.

In [3]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier

forest_name, svm_name = "Random forest", "RBF SVM C=10"
forest_pred, svm_pred = predictions[forest_name], predictions[svm_name]
forest_ok, svm_ok = forest_pred == yh, svm_pred == yh
print("Both correct:", (forest_ok & svm_ok).sum(), "Both wrong:", (~forest_ok & ~svm_ok).sum())
display(paired_by(yh, forest_pred, svm_pred, yh))
rescued = pd.DataFrame({"actual": yh, "forest_called": forest_pred, "group": groups})[~forest_ok & svm_ok]
display(rescued.groupby(["actual", "forest_called"]).agg(
    svm_rescues=("group","size"), distinct_readings=("group","nunique")
).sort_values("svm_rescues",ascending=False).head(12))
tennis = (yh == "Tennis") & (forest_pred == "Football")
print("Tennis called Football:", tennis.sum(), "distinct readings:", np.unique(groups[tennis]).size,
      "rescued by SVM:", svm_ok[tennis].sum())

def grouped_splits(rows):
    target, identities = yh[rows], groups[rows]
    for n in (3, 2):
        for seed in range(SEED, SEED+200):
            cv = StratifiedGroupKFold(n, shuffle=True, random_state=seed)
            splits = list(cv.split(Xh.iloc[rows], target, identities))
            if all(len(np.unique(target[a])) == len(labels) == len(np.unique(target[b]))
                   for a,b in splits):
                return splits
    raise ValueError("No grouped split covers every activity.")

def activity_blend(rows, weights=(2,1)):
    calibrated = CalibratedClassifierCV(clone(har_models[svm_name]), method="sigmoid",
                                        cv=grouped_splits(rows), ensemble=False)
    return VotingClassifier([("forest",clone(har_models[forest_name])), ("svm",calibrated)],
                            voting="soft", weights=list(weights), flatten_transform=False)

# First candidate wins an exact inner accuracy tie.
weight_grid = [(2,1), (1,1), (1,2), (3,1), (1,3)]
def tune_weights(rows):
    pred_grid = np.empty((len(weight_grid), len(rows)), dtype=yh.dtype)
    for a,b in grouped_splits(rows):
        train, valid = rows[a], rows[b]
        candidate = activity_blend(train).fit(Xh.iloc[train],yh[train])
        forest_p, svm_p = candidate.transform(Xh.iloc[valid])
        for i,(f,s) in enumerate(weight_grid):
            pred = candidate.classes_[np.argmax(f*forest_p + s*svm_p,axis=1)]
            pred_grid[i,b] = pred
    correct = [(pred == yh[rows]).sum() for pred in pred_grid]
    macro = [f1_score(yh[rows],pred,labels=labels,average="macro",zero_division=0) for pred in pred_grid]
    return weight_grid[np.argmax(correct)], weight_grid[np.argmax(macro)]

blend_name = "Forest + calibrated SVM (2:1)"
tuned_name = "Forest + calibrated SVM (inner-selected weights)"
blend_pred, tuned_pred = np.empty_like(yh), np.empty_like(yh)
macro_name = "Forest + calibrated SVM (inner macro-F1 weights)"
macro_pred = np.empty_like(yh)
fold_number = np.zeros(len(yh), dtype=int)
for fold, (tr, va) in enumerate(har_folds, start=1):
    weights, macro_weights = tune_weights(tr)
    fit = activity_blend(tr).fit(Xh.iloc[tr],yh[tr])
    blend_pred[va] = fit.predict(Xh.iloc[va])
    fit.weights = list(weights)
    tuned_pred[va] = fit.predict(Xh.iloc[va])
    fit.weights = list(macro_weights)
    macro_pred[va] = fit.predict(Xh.iloc[va])
    fold_number[va] = fold
    print("Outer fold", fold, "inner-selected forest:SVM weights", weights, "macro-F1 weights", macro_weights)
print("Fixed blend versus forest")
display(paired_by(yh, forest_pred, blend_pred, fold_number))
print("Tuned versus fixed blend")
display(paired_by(yh, blend_pred, tuned_pred, fold_number))
print("Macro-F1-tuned blend versus fixed blend (row counts)")
display(paired_by(yh, blend_pred, macro_pred, fold_number))
for name, pred in [(blend_name,blend_pred), (tuned_name,tuned_pred), (macro_name,macro_pred)]:
    predictions[name] = pred
    har_results.append(dict(model=name,correct=(pred == yh).sum(),accuracy=accuracy_score(yh,pred),
                            macro_f1=f1_score(yh,pred,average="macro")))
har_board = pd.DataFrame(har_results).sort_values(["correct","macro_f1"],ascending=False,kind="stable")
display(har_board)
har_winner = har_board.iloc[0].model
print("Activity winner:", har_winner)
display(pd.DataFrame(confusion_matrix(yh,predictions[har_winner],labels=labels),index=labels,columns=labels))

Both correct: 1621 Both wrong: 415
                   rows  fixed  lost  net   delta_pp
All                2442    184   222  -38  -1.556102
Tennis              580     66    73   -7  -1.206897
Sitting             696      4    11   -7  -1.005747
Cycling             274      9    24  -15  -5.474453
Football            387     18    61  -43 -11.111111
Walking              59      8     3    5   8.474576
Swimming            122     13    29  -16 -13.114754
WalkingUpstairs      94     24     4   20  21.276596
WalkingDownstairs    84     13     1   12  14.285714
JumpRope            117     26    15   11   9.401709
Jogging              12      0     1   -1  -8.333333
Pushups              17      3     0    3  17.647059
                                 svm_rescues  distinct_readings
actual            forest_called                                
Tennis            Football                54                 54
JumpRope          Tennis                  22                  5
Football          Te

## Refit and save
The bundle contains the selected pipelines, threshold and dated transaction fields used by the fraud feature builder. Treat it as private course data. Keep this notebook with the bundle. The functions below use the same feature rules for future CSV rows. Optional download is off by default.

In [7]:
output = Path("ps5_accuracy_models")
output.mkdir(exist_ok=True)
fraud_bundle = dict(model=fit_fraud(fraud_winner,np.arange(len(yf))), name=fraud_winner,
                    columns=cols, threshold=thresholds[fraud_winner], sklearn_version=sklearn.__version__)
history_needed = any(c not in basic for c in cols)
if history_needed:
    fraud[event_fields].to_pickle(output/"history.pkl.gz",compression="gzip")
joblib.dump(fraud_bundle,output/"fraud.joblib")
if har_winner in (blend_name, tuned_name, macro_name):
    rows = np.arange(len(yh))
    weights = (2,1) if har_winner == blend_name else tune_weights(rows)[int(har_winner == macro_name)]
    print("Export forest:SVM weights:", weights)
    activity_final = activity_blend(rows, weights)
else:
    activity_final = clone(har_models[har_winner])
joblib.dump(activity_final.fit(Xh,yh),output/"activity.joblib")

def predict_fraud_rows(rows, folder=output):
    bundle = joblib.load(folder/"fraud.joblib")
    if any(c not in basic for c in bundle["columns"]):
        history = pd.read_pickle(folder/"history.pkl.gz")
        joined = pd.concat([history,rows[event_fields]],ignore_index=True)
        features = fraud_features(joined).iloc[len(history):]
    else:
        features = pd.DataFrame({"amount":rows.amt, "hour":rows.trans_time.str.slice(0,2).astype(int)})
    scores = fraud_scores(bundle["model"],features,bundle["columns"])
    return (scores >= bundle["threshold"]).astype(int), scores

def predict_activity_rows(predictors, folder=output):
    return joblib.load(folder/"activity.joblib").predict(predictors)

# For incoming fraud CSVs use pd.read_csv(path, dtype={k:"string" for k in entity_fields}).
# For activity CSVs use header=None and pass the first Xh.shape[1] columns.
print("Saved:", fraud_winner, "and", har_winner, "in", output.resolve())
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files
    files.download(shutil.make_archive(str(output),"zip",output))

Export forest:SVM weights: (2, 1)
Saved: Random forest · extended + spending trend and Forest + calibrated SVM (2:1) in ps5_accuracy_models
